# preTextAnalysis: de acciones seleccionadas a un corpus
Doctorado en Ciencia Política — UNMSM · Métodos Cuantitativos (2026)

**Puzzle:** los partidos pueden compartir prioridades económicas y proponer acciones diferentes. ¿Hasta qué punto sus propuestas se parecen y qué orientación expresan sobre el papel del Estado?

**H1:** similitud entre las propuestas de los partidos. **H2:** diferenciación en una dimensión económica estatal-redistributiva / liberal-mercado.

Pregunta → sección relevante → selección manual de acciones → corpus → representación → análisis.

La **unidad de análisis es el partido, representado por el conjunto de sus acciones seleccionadas**. Cada partido ocupa una sola fila. Este notebook prepara la evidencia; todavía no calcula embeddings ni evalúa las hipótesis.

## Antes de Python: leer y seleccionar

Los ocho PDFs representan una sección ya elegida manualmente de un plan más amplio: **Economía y papel del Estado**. Son documentos ficticios para aprender el procedimiento. Cada uno contiene introducción, diagnóstico, acciones propuestas y conclusión.

Lea toda la sección para comprender su contexto. Luego copie **solo las viñetas de “Acciones propuestas”**, completas y en su orden original. No resuma, reformule ni elimine negaciones o condiciones. Introducción, diagnóstico y conclusión no entran al corpus.

Complete `plantilla_acciones.csv`: **todas las acciones de un partido van en una misma celda de `texto_original`**, separadas por saltos de línea o espacios. La plantilla tiene ocho filas, una por partido. No cree una fila por viñeta. Conserve identificador, nombre, candidatura, sección y página de origen.

Guarde el archivo como **`acciones_seleccionadas.csv`**, en CSV UTF-8. En Colab, cargue únicamente ese archivo con el panel Archivos. No necesita cargar los PDFs ni convertirlos a LaTeX. El archivo resuelto en `docente/` sirve para cotejar la selección o demostrar el notebook.

**Clave:** los partidos pueden tener distintas cantidades de propuestas. No añadimos ni retiramos acciones para igualarlas. En estos PDFs hay seis por partido, pero esa cantidad no es un requisito del corpus.

## preTEXT-01: leer la selección manual

Leemos un único CSV y creamos el DataFrame `corpus`. Cada fila representa un partido y contiene todas sus acciones seleccionadas en `texto_original`.

Usamos `pandas`, normalmente disponible en Colab. No instalamos lectores de PDF ni modelos en este notebook.

In [ ]:
import pandas as pd

corpus = pd.read_csv('acciones_seleccionadas.csv', keep_default_na=False)

corpus

## preTEXT-02: reconocer la estructura del corpus

El corpus debe tener ocho filas, una por partido. Revisamos los identificadores y la procedencia de la sección elegida. La cantidad de acciones dentro de cada texto puede variar sin cambiar esta estructura.

In [ ]:
print('Total de partidos:', len(corpus))

corpus[['id_partido', 'partido', 'seccion', 'pagina']]

## preTEXT-03: preparar el formato del texto

La selección manual ya excluyó introducciones, diagnósticos y conclusiones. Aquí solo uniformamos caracteres, retiramos el guion opcional y unimos los espacios o saltos de línea producidos al copiar.

Guardamos el resultado en `texto` y conservamos `texto_original`. Cada fila sigue conteniendo todas las acciones de un partido. Mantenemos puntuación, acentos, mayúsculas, negaciones y condiciones; no eliminamos stopwords ni resumimos.

**Intuición:** “reducir impuestos” y “reducir impuestos sin aumentar el déficit” no expresan exactamente el mismo compromiso. La condición forma parte de la evidencia.

In [ ]:
import re
import unicodedata

def limpiar(texto):
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.replace('\u00ad', '')
    return re.sub(r'\s+', ' ', texto).strip()

corpus['texto'] = corpus['texto_original'].apply(limpiar)

corpus[['id_partido', 'texto']]

## preTEXT-04: comparar y volver a la fuente

Mostramos el conjunto de acciones original y preparado de cada partido. Compruebe que la limpieza solo modificó el formato y que todas las propuestas siguen presentes.

Para revisar la selección manual, vuelva al PDF correspondiente: esta comparación no detecta por sí sola una viñeta omitida o copiada incorrectamente. Si encuentra errores, corrija el CSV de entrada y ejecute de nuevo los bloques.

**Cuidado:** un texto bien formateado no garantiza una selección correcta.

In [ ]:
for _, fila in corpus.iterrows():
    print(f"\n{fila['id_partido']} — {fila['partido']} — página {fila['pagina']}")
    print('ORIGINAL:\n', fila['texto_original'])
    print('PREPARADO:\n', fila['texto'])
    print('-' * 80)

## preTEXT-05: guardar un único corpus preparado

Guardamos **`acciones_preparadas.csv`**, con una fila por partido y las columnas originales más `texto`. La comprobación final evita exportar una plantilla sin completar o identificadores duplicados.

Este CSV será la entrada de `TextAnalysis`. Para representar directamente cada partido con un embedding, allí tendremos que comprobar que **su texto completo** cabe en el modelo elegido. El número de propuestas puede variar; lo que limita la entrada es su longitud en tokens. Preparar este CSV no resuelve automáticamente esa comprobación.

En Colab, descargue el archivo si desea conservarlo o utilizarlo en otra sesión. No se crean archivos adicionales.

In [ ]:
assert len(corpus) > 0 and corpus['texto'].ne('').all(), 'Complete los textos antes de guardar.'
assert corpus['id_partido'].is_unique, 'Debe haber una sola fila por partido.'

corpus.to_csv('acciones_preparadas.csv', index=False, encoding='utf-8')

print('Corpus guardado: acciones_preparadas.csv')

## Qué no puede responder este corpus

Cada partido está representado por las acciones de una sección seleccionada. El corpus no representa todo su programa, toda su ideología ni las políticas que realmente implementará. Tampoco permite explicar por sí solo por qué dos partidos proponen medidas similares.

Antes de pasar al análisis, justifique por qué esta sección es pertinente para H1 y H2 y qué información quedó fuera. **La selección de evidencia precede a la técnica.**